# fedstat — проверка и примеры

Демонстрация библиотеки [`fedstat`](https://pypi.org/project/fedstat/) на реальных показателях ЕМИСС.
Каждый пример: посмотреть доступные фильтры → задать нужные → скачать `DataFrame` → «широкий» вид.

> Требуется доступ к fedstat.ru (российский IP). Установка: `pip install -U fedstat`.

In [ ]:
import fedstat
import pandas as pd
pd.set_option('display.max_colwidth', None)

## Поиск индикатора по названию

`find` ищет показатель по названию (по **подстрока**, без морфологии — ищи по основе слова: «ипотеч», «безработиц»; несколько слов = И). Возвращает `id | title | department | hidden`. `catalog()` — весь список индикаторов ЕМИСС.

In [ ]:
fedstat.find("цена жил")      # средняя цена жилья и смежные

In [ ]:
fedstat.find("ипотеч кредит") # несколько слов: все должны встретиться

In [ ]:
fedstat.catalog().shape       # размер полного каталога (кол-во индикаторов, колонок)

## Как узнать, какие фильтры доступны

`filter_options(id)` — обзор всех полей и их уникальных значений: словарём `{поле: [значения]}` или таблицей (`as_frame=True`). Заменяет ручной перебор через `.unique()`.

In [ ]:
fedstat.filter_options("31452", as_frame=True)

In [ ]:
# только конкретное поле:
fedstat.filter_options("31452")["Типы квартир"]

> **Про «широкий» вид (`to_wide`).** Это `pivot_table` со средним по умолчанию. В таблицу попадают только `index` и `columns`; **все прочие измерения (регион, рынок и т.п.) усредняются**. Поэтому для осмысленного ряда сначала фильтруем до одного региона (ниже — по РФ). Чтобы оставить все регионы, добавь их в `index`: `to_wide(df, columns=..., index=["region", "year"])`.

## Пример 1 — Средняя цена 1 кв. м жилья (31452)

Кварталы, по годам и рынкам.

In [ ]:
f = fedstat.filter_template("31452")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Рынок жилья"] = ["Первичный рынок жилья", "Вторичный рынок жилья"]
f["Типы квартир"] = ["Все типы квартир"]

OKATO = "Классификатор объектов административно-территориального деления (ОКАТО)"
df_price = (
    fedstat.load("31452", filters=f)
    .rename(columns={OKATO: "region", "Рынок жилья": "market",
                     "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "market", "period", "year", "value"]]
)
df_price.head()

In [ ]:
# «широкий» вид ПО РФ, первичный рынок (иначе усреднится по всем регионам)
rf = df_price[(df_price.region.isin(["Российская Федерация", "Российская Федерация без учета новых субъектов (с 01.01.2023)"])) & (df_price.market == "Первичный рынок жилья")]
fedstat.to_wide(rf, columns="period", values="value", index="year")

In [ ]:
fedstat.to_wide(df_price, columns="period", values="value", index=["region","market","year"])

## Пример 2 — Индекс цен на жильё (30925)

Есть поле «Виды показателя» — выбираем один вид.

In [ ]:
fedstat.filter_options("30925")["Виды показателя"]

In [ ]:
f = fedstat.filter_template("30925")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Виды показателя"] = ["К соответствующему кварталу предыдущего года"]
f["Типы квартир"] = ["Все типы квартир"]

df_hpi = (
    fedstat.load("30925", filters=f)
    .rename(columns={OKATO: "region", "Рынок жилья": "market",
                     "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "market", "period", "year", "value"]]
)
df_hpi.head()

In [ ]:
rf = df_hpi[(df_hpi.region == "Российская Федерация") & (df_hpi.market == "Первичный рынок жилья")]
fedstat.to_wide(rf, columns="period", values="value", index="year")

## Пример 3 — Уровень безработицы (43062)

Есть поле «Возраст».

In [ ]:
fedstat.filter_options("43062")["Возраст"]

In [ ]:
f = fedstat.filter_template("43062")
f["Год"] = [str(y) for y in range(2017, 2027)]
f["Возраст"] = ["15 лет и старше"]
f["Период"] = ["I квартал", "II квартал", "III квартал", "IV квартал"]

df_unemp = (
    fedstat.load("43062", filters=f)
    .rename(columns={OKATO: "region", "PERIOD": "period", "TIME": "year", "VALUE": "value"})
    [["region", "period", "year", "value"]]
)
df_unemp.head()

In [ ]:
rf = df_unemp[df_unemp.region == "Российская Федерация"]
fedstat.to_wide(rf, columns="period", values="value", index="year")

## Пример 4 — Ставки по ипотеке (59345, месячные данные)

**Месячная** периодика и иерархические регионы.

In [ ]:
fedstat.filter_options("59345")["Характеристики кредита"]

In [ ]:
f = fedstat.filter_template("59345")
f["Год"] = [str(y) for y in range(2018, 2027)]
f["Характеристики жилищных/ипотечных кредитов"] = ["Средневзвешенная ставка по кредитам, выданным в течение месяца"]
f["Характеристики кредита"] = ["Ипотечные жилищные кредиты"]

REGION = "Регионы Российской Федерации (иерархический)"
df_rate = (
    fedstat.load("59345", filters=f)
    .rename(columns={REGION: "region", "PERIOD": "month", "TIME": "year", "VALUE": "value"})
    [["region", "month", "year", "value"]]
)
df_rate.head()

In [ ]:
rf = df_rate[df_rate.region == "Российская Федерация"]
fedstat.to_wide(rf, columns="month", values="value", index="year")